In [4]:
import pandas as pd
import numpy as np
import subprocess
import sys
import mlflow

In [5]:
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')

In [6]:
import mlflow

# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment("Projeto_Churn_Tech_Challenge")


MlflowException: Cannot set a deleted experiment 'Projeto_Churn_Tech_Challenge' as the active experiment. You can restore the experiment, or permanently delete the experiment to create a new one.

In [ ]:
mlflow.get_tracking_uri()

'sqlite:///../mlflow.db'

# --------------- REGRESSÃO LOGISTICA ---------------

In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.data
import matplotlib.pyplot as plt # Adicionado para o gráfico
import os # Adicionado para gerenciar arquivos
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             classification_report, ConfusionMatrixDisplay) # Adicionado ConfusionMatrixDisplay

# 1. Configurações de separação
test_size = 0.2
random_state = 42

# Lista de colunas que entregam a resposta ou que não devem ser usadas como features
cols_para_remover = ['churn_value', 'churn_reason', 'count', 'country', 'state']

# Separando variáveis alvo e features com segurança
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

# Split com estratificação (importante para Churn!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

# 2. Criando o objeto de dataset do MLflow (para a aba "Inputs")
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# Preprocessamento
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Aplicando transformação
X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)

feature_names = preprocessador.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_transformado, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformado, columns=feature_names)

# --- INÍCIO DO EXPERIMENTO NO MLFLOW ---
# Nome amigável para a sua Run
run_name_input = "regressao_logistica_baseline"

with mlflow.start_run(run_name=run_name_input) as run:
    
    # Definindo o modelo
    model = LogisticRegression(random_state=random_state, max_iter=1000)
    
    # 1. Logando o Dataset (Aba Inputs)
    mlflow.log_input(dataset, context="training")
    
    # 2. Logando Parâmetros (Aba Parameters)
    mlflow.log_params({
        "model_type": "LogisticRegression",
        "max_iter": 1000,
        "random_state": random_state,
        "test_size": test_size,
        "stratify": "True",
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Treinando o modelo
    model.fit(X_train_df, y_train)

    # 4. Logando o modelo treinado (Aba Artifacts)
    # É melhor logar DEPOIS do fit para salvar os pesos
    mlflow.sklearn.log_model(model, "logistic_regression_model")

    # Predições e métricas
    y_pred_test = model.predict(X_test_df)
    y_probs = model.predict_proba(X_test_df)[:, 1]

    # Matriz de Confusão numérica
    cm = confusion_matrix(y_test, y_pred_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred_test),
        "precision": precision_score(y_test, y_pred_test),
        "recall": recall_score(y_test, y_pred_test),
        "f1_score": f1_score(y_test, y_pred_test),
        "roc_auc": roc_auc_score(y_test, y_probs)
    }

    # 5. Logando Métricas (Aba Metrics)
    mlflow.log_metrics(metrics)

    # 6. Gerando e Logando o Gráfico da Matriz de Confusão (Aba Artifacts)
    # Define o nome do arquivo temporário
    plot_file_name = "confusion_matrix.png"
    
    # Gera o gráfico
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Blues')
    ax.set_title(f'Matriz de Confusão - {run_name_input}')
    
    # Salva o gráfico localmente
    plt.savefig(plot_file_name)
    plt.close(fig) # Fecha o plot para liberar memória
    
    # Salva o gráfico como artefato no MLflow
    mlflow.log_artifact(plot_file_name)
    
    # (Opcional) Remove o arquivo temporário local
    if os.path.exists(plot_file_name):
        os.remove(plot_file_name)

    # Output no console
    print(f"Logistic Regression - Métricas de validação")
    print("-" * 30)
    for k, v in metrics.items():
        print(f"{k.capitalize()}: {v:.4f}")
    print("-" * 30)
    print("Confusion Matrix (Numeric):\n", cm)
    print("Classification Report:\n", classification_report(y_test, y_pred_test))

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

Logistic Regression - Métricas de validação
------------------------------
Accuracy: 0.7970
Precision: 0.6310
Recall: 0.5668
F1_score: 0.5972
Roc_auc: 0.8433
------------------------------
Confusion Matrix (Numeric):
 [[911 124]
 [162 212]]
Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.88      0.86      1035
           1       0.63      0.57      0.60       374

    accuracy                           0.80      1409
   macro avg       0.74      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409



# --------------- DUMMY CLASSIFIER ---------------

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix


# Criando o objeto de dataset do MLflow (para a aba "Inputs")
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# =========================
# Configurações e Split
# =========================
SEED = 20
test_size = 0.2


# Lista de colunas que entregam a resposta ou que não devem ser usadas como features
cols_para_remover = ['churn_value', 'churn_reason', 'count', 'country', 'state']

# Separando variáveis alvo e features com segurança
X = df.drop(columns=cols_para_remover)
y = df['churn_value']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)

# =========================
# Preprocessamento
# =========================
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Aplicando transformação nos dados para o fit manual (fora do pipeline se necessário)
X_train_transformed = preprocessador.fit_transform(X_train)
X_test_transformed = preprocessador.transform(X_test)

# =========================
# Início do MLflow
# =========================
with mlflow.start_run(run_name="Dummy_Classifier_baseline"):

    # 1. Definindo o Modelo
    modelo_dummy = DummyClassifier(strategy='stratified', random_state=SEED)

    mlflow.log_input(dataset, context="training")  # Logando o dataset na aba Inputs

    # 2. Log de Parâmetros
    mlflow.log_params({
        "model_type": "DummyClassifier",
        "random_state": SEED,
        "test_size": test_size,
        "stratify": "True",
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Treino (Fit)
    # Importante: o fit deve vir antes do log_model para salvar o estado do modelo
    modelo_dummy.fit(X_train_transformed, y_train)

    # 4. Log do Modelo
    mlflow.sklearn.log_model(modelo_dummy, "dummy_classifier_model")

    # 5. Predição e ajuste de Limiar (Exemplo com 0.5 padrão)
    threshold = 0.5
    y_probs = modelo_dummy.predict_proba(X_test_transformed)[:, 1]
    y_pred = (y_probs >= threshold).astype(int)

    # 6. Cálculo e Log de Métricas (Média da Validação Cruzada)
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    
    # Criamos um pipeline apenas para a validação cruzada para evitar data leakage
    from sklearn.pipeline import Pipeline
    pipeline_dummy = Pipeline([('pre', preprocessador), ('model', modelo_dummy)])
    
    resultados_cv = cross_validate(
        pipeline_dummy, X_train, y_train, cv=cv, 
        scoring=['accuracy', 'precision', 'recall', 'f1']
    )

    metrics = {
        "accuracy": resultados_cv['test_accuracy'].mean(),
        "precision": resultados_cv['test_precision'].mean(),
        "recall": resultados_cv['test_recall'].mean(),
        "f1_score": resultados_cv['test_f1'].mean()
    }
    mlflow.log_metrics(metrics)

    # 7. Matriz de Confusão (Artefato)
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Greys')
    ax.set_title('Matriz de Confusão - Dummy Classifier')
    
    plot_path = "confusion_matrix_dummy.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

    print("===== Dummy Classifier Gravado no MLflow =====")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

===== Dummy Classifier Gravado no MLflow =====
accuracy: 0.6006
precision: 0.2624
recall: 0.2789
f1_score: 0.2704


# --------------- ÁRVORE DE DECISÃO ---------------

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             classification_report, ConfusionMatrixDisplay)

# --- CONFIGURAÇÕES GERAIS ---
run_name_dt = "decision_tree_baseline   "
max_depth = 5 
random_state = 42
test_size = 0.2


# Lista de colunas que entregam a resposta ou que não devem ser usadas como features
cols_para_remover = ['churn_value', 'churn_reason', 'count', 'country', 'state']

# Separando variáveis alvo e features com segurança
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

# 2. Configuração do Tracking de Dados no MLflow
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# 3. Preprocessamento
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)

feature_names = preprocessador.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_transformado, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformado, columns=feature_names)

# --- EXECUÇÃO DO EXPERIMENTO ---
with mlflow.start_run(run_name=run_name_dt) as run:
    
    # 1. Definição e Treino
    model_dt = DecisionTreeClassifier(max_depth=max_depth, random_state=random_state)
    model_dt.fit(X_train_df, y_train)
    
    # 2. Log de Inputs e Parâmetros
    mlflow.log_input(dataset, context="training")
    mlflow.log_params({
        "model_type": "DecisionTreeClassifier",
        "max_depth": max_depth,
        "random_state": random_state,
        "test_size": test_size,
        "stratify": "True",
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Log do Modelo (Artefato)
    mlflow.sklearn.log_model(model_dt, "decision_tree_model")

    # 4. Predições e Métricas
    y_pred_dt = model_dt.predict(X_test_df)
    y_probs_dt = model_dt.predict_proba(X_test_df)[:, 1]

    metrics_dt = {
        "accuracy": accuracy_score(y_test, y_pred_dt),
        "precision": precision_score(y_test, y_pred_dt),
        "recall": recall_score(y_test, y_pred_dt),
        "f1_score": f1_score(y_test, y_pred_dt),
        "roc_auc": roc_auc_score(y_test, y_probs_dt)
    }
    mlflow.log_metrics(metrics_dt)

    # 5. Matriz de Confusão (Gráfico - Artefato)
    cm_dt = confusion_matrix(y_test, y_pred_dt)
    plot_cm_path = "confusion_matrix_dt.png"
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Greens')
    ax.set_title(f'Matriz de Confusão - {run_name_dt}')
    
    plt.savefig(plot_cm_path)
    plt.close(fig)
    mlflow.log_artifact(plot_cm_path)
    if os.path.exists(plot_cm_path): os.remove(plot_cm_path)

    # 6. Feature Importance (Gráfico - Artefato)
    # Extrai as importâncias e associa aos nomes das features
    importances = model_dt.feature_importances_
    # Criamos um DataFrame para ordenar as TOP features
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values(by='importance', ascending=False)
    
    # Pegamos apenas as 15 mais importantes para não poluir o gráfico
    top_features = feature_importance_df.head(15)
    
    # Gera o gráfico de barras horizontais
    plot_fi_path = "feature_importance_dt.png"
    fig_fi, ax_fi = plt.subplots(figsize=(10, 8))
    # Inverte a ordem para a mais importante ficar no topo
    ax_fi.barh(top_features['feature'][::-1], top_features['importance'][::-1], color='seagreen')
    ax_fi.set_xlabel('Importância (Ponderada)')
    ax_fi.set_title(f'Top 15 Variáveis Mais Decisivas para o Churn - {run_name_dt}')
    ax_fi.grid(axis='x', linestyle='--', alpha=0.5)
    
    plt.savefig(plot_fi_path)
    plt.close(fig_fi)
    
    # Loga o gráfico como artefato
    mlflow.log_artifact(plot_fi_path)
    if os.path.exists(plot_fi_path): os.remove(plot_fi_path)

    # (Opcional) Logar a tabela de importância como CSV
    csv_fi_path = "feature_importance_table.csv"
    feature_importance_df.to_csv(csv_fi_path, index=False)
    mlflow.log_artifact(csv_fi_path)
    if os.path.exists(csv_fi_path): os.remove(csv_fi_path)

    # 7. Relatório Final
    print(f"Decision Tree - Rastreamento finalizado no MLflow.")
    print("-" * 30)
    print("Métricas:")
    for k, v in metrics_dt.items():
        print(f"  {k.capitalize()}: {v:.4f}")
    print("-" * 30)
    print("TOP 5 Variáveis Mais Importantes:")
    print(feature_importance_df.head(5))

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

Decision Tree - Rastreamento finalizado no MLflow.
------------------------------
Métricas:
  Accuracy: 0.7842
  Precision: 0.6115
  Recall: 0.5134
  F1_score: 0.5581
  Roc_auc: 0.8335
------------------------------
TOP 5 Variáveis Mais Importantes:
                                feature  importance
1162       cat__contract_Month-to-month    0.474801
1171                 num__tenure_months    0.162974
1142  cat__internet_service_Fiber optic    0.153993
1134                 cat__dependents_No    0.079731
1153               cat__tech_support_No    0.022646


# --------------- RANDOM FOREST ---------------

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
from sklearn.ensemble import RandomForestClassifier # O novo modelo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             ConfusionMatrixDisplay)

# --- CONFIGURAÇÕES DO RANDOM FOREST ---
run_name_rf = "random_forest_baseline"
n_estimators = 100  # Número de árvores na floresta
max_depth = 10      # Profundidade para evitar overfitting
random_state = 42
test_size = 0.2

# 1. Preparação dos Dados (Mantendo a consistência)
# Lista de colunas que entregam a resposta ou que não devem ser usadas como features
cols_para_remover = ['churn_value', 'churn_reason', 'count', 'country', 'state']

# Separando variáveis alvo e features com segurança
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

# 2. Tracking do Dataset
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# 3. Preprocessamento
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)
feature_names = preprocessador.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_transformado, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformado, columns=feature_names)

# --- EXECUÇÃO DO EXPERIMENTO ---
with mlflow.start_run(run_name=run_name_rf) as run:
    
    mlflow.log_input(dataset, context="training")
    # 1. Definição do Modelo (Random Forest)
    model_rf = RandomForestClassifier(
        n_estimators=n_estimators, 
        max_depth=max_depth, 
        random_state=random_state,
        n_jobs=-1 # Usa todos os núcleos do seu PC para treinar mais rápido
    )
    
    model_rf.fit(X_train_df, y_train)
    
    # 2. Log de Parâmetros Específicos do RF
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": random_state,
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Log do Modelo
    mlflow.sklearn.log_model(model_rf, "random_forest_model")

    # 4. Predições e Métricas
    y_pred_rf = model_rf.predict(X_test_df)
    y_probs_rf = model_rf.predict_proba(X_test_df)[:, 1]

    metrics_rf = {
        "accuracy": accuracy_score(y_test, y_pred_rf),
        "precision": precision_score(y_test, y_pred_rf),
        "recall": recall_score(y_test, y_pred_rf),
        "f1_score": f1_score(y_test, y_pred_rf),
        "roc_auc": roc_auc_score(y_test, y_probs_rf)
    }
    mlflow.log_metrics(metrics_rf)

    # 5. Artefato: Matriz de Confusão
    cm_rf = confusion_matrix(y_test, y_pred_rf)
    plot_cm_path = "confusion_matrix_rf.png"
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Purples')
    ax.set_title(f'Matriz de Confusão - {run_name_rf}')
    plt.savefig(plot_cm_path)
    plt.close()
    mlflow.log_artifact(plot_cm_path)

    # 6. Artefato: Feature Importance
    importances = model_rf.feature_importances_
    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values(by='importance', ascending=False)
    
    plot_fi_path = "feature_importance_rf.png"
    fig_fi, ax_fi = plt.subplots(figsize=(10, 8))
    ax_fi.barh(fi_df.head(15)['feature'][::-1], fi_df.head(15)['importance'][::-1], color='rebeccapurple')
    ax_fi.set_title('Top 15 Variáveis - Random Forest')
    plt.savefig(plot_fi_path)
    plt.close()
    mlflow.log_artifact(plot_fi_path)

    print(f"Random Forest registrado! AUC: {metrics_rf['roc_auc']:.4f}")

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

Random Forest registrado! AUC: 0.8382


========================= COMPARACAO DE METRICAS ENTRE MODELOS + ANALISE DE CUSTOS ==========================

In [21]:
import mlflow
import pandas as pd

# --- PREMISSAS DE NEGÓCIO ATUALIZADAS ---
TAMANHO_BASE_TESTE = 1409
PREVALENCIA_CHURN = 0.265
VALOR_MEDIO_TOTAL = 2280.00  # Impacto de perder o cliente (TotalCharges)
VALOR_MEDIO_MENSAL = 64.70    # Custo de tentar reter (MonthlyCharges)

experiment_name = "Projeto_Churn_Tech_Challenge"
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment:
    # 1. Busca todas as runs
    df_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    
    # 2. Limpeza de colunas e remoção de NaNs
    df_runs.columns = [c.replace("metrics.", "").replace("params.", "").replace("tags.mlflow.", "") for c in df_runs.columns]
    
    # Filtramos apenas runs que completaram as métricas e não são "viciadas" (Recall < 0.95 para ser realista)
    df_clean = df_runs.dropna(subset=['recall', 'precision', 'model_type'])
    df_clean = df_clean[df_clean['recall'] < 0.95].copy() 

    # --- 3. CÁLCULO FINANCEIRO DETALHADO ---
    def calcular_financeiro(row):
        n_churners = TAMANHO_BASE_TESTE * PREVALENCIA_CHURN
        
        # FN: Perda de receita acumulada
        fn_count = n_churners * (1 - row['recall'])
        impacto_fn = fn_count * VALOR_MEDIO_TOTAL
        
        # FP: Desperdício de marketing mensal
        total_pos = n_churners / row['precision'] if row['precision'] > 0 else 0
        fp_count = max(0, total_pos - (n_churners * row['recall']))
        impacto_fp = fp_count * VALOR_MEDIO_MENSAL
        
        return pd.Series([impacto_fn, impacto_fp, impacto_fn + impacto_fp])

    df_clean[['impacto_fn', 'impacto_fp', 'custo_total']] = df_clean.apply(calcular_financeiro, axis=1)

    # --- 4. AGRUPAMENTO: UMA LINHA POR MODELO ---
    # Pegamos a run com o MENOR custo total para cada tipo de modelo
    leaderboard = df_clean.sort_values("roc_auc", ascending=True).drop_duplicates("model_type")

    # 5. Seleção e Formatação
    cols_finais = ["model_type", "accuracy", "recall", "precision","roc_auc", "impacto_fn", "impacto_fp", "custo_total"]
    leaderboard = leaderboard[cols_finais].sort_values("custo_total")

    # Formatação para Moeda
    for col in ['impacto_fn', 'impacto_fp', 'custo_total']:
        leaderboard[col] = leaderboard[col].apply(lambda x: f"R$ {x:,.2f}")

    print("===== LEADERBOARD FINAL: IMPACTO NO NEGÓCIO =====")
    display(leaderboard)

===== LEADERBOARD FINAL: IMPACTO NO NEGÓCIO =====


,model_type,accuracy,recall,precision,roc_auc,impacto_fn,impacto_fp,custo_total
6,LogisticRegression,0.797019,0.566845,0.630952,0.843323,"R$ 368,752.63","R$ 24,594.32","R$ 393,346.95"
2,DecisionTreeClassifier,0.784244,0.513369,0.611465,0.833534,"R$ 414,277.65","R$ 27,106.44","R$ 441,384.08"
0,RandomForestClassifier,0.784954,0.344920,0.689840,0.838159,"R$ 557,681.45","R$ 26,687.17","R$ 584,368.62"
5,DummyClassifier,0.600635,0.278886,0.262443,NaN,"R$ 613,897.26","R$ 85,313.08","R$ 699,210.34"
